# Native CoordGen aligned depiction

Helpers: `notebooks/_coordgen_align.py` (Chematic parse + Rust cost map +
soft-constrained CoordGen). Tests: `tests/test_coordgen_align_notebook.py`.

- No RDKit, no overlays — side-by-side painted SVGs only
- Columns: **ref free** | **query free** | **query constrained**
- Key row: hexane → cyclohexane (soft ring stays hexagonal)

```bash
make build-python
uv run pytest -q tests/test_coordgen_align_notebook.py tests/test_chematic_align.py
```


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

from IPython.display import HTML, display

_NB = Path.cwd()
if (_NB / "_coordgen_align.py").is_file():
    sys.path.insert(0, str(_NB))
elif (_NB / "notebooks" / "_coordgen_align.py").is_file():
    sys.path.insert(0, str(_NB / "notebooks"))
else:
    raise RuntimeError("run from repo root or notebooks/")

import _coordgen_align as _cga
importlib.reload(_cga)

DEFAULT_PAIRS = _cga.DEFAULT_PAIRS
native_aligned = _cga.native_aligned
matched_rms = _cga.matched_rms
paint_svg = _cga.paint_svg
side_by_side_svgs = _cga.side_by_side_svgs
hexagon_rel_dev = _cga.hexagon_rel_dev


def show_row(title: str, panels: list[tuple[str, str]]):
    bits = [f"<h3>{title}</h3><div style='display:flex;gap:1rem;flex-wrap:wrap'>"]
    for label, svg in panels:
        bits.append(
            f"<div><div style='font:12px sans-serif;margin-bottom:4px'>{label}</div>{svg}</div>"
        )
    bits.append("</div>")
    display(HTML("".join(bits)))


In [ ]:
for pair in DEFAULT_PAIRS:
    ref, query, ref_free, query_free, soft, pairs = native_aligned(pair.ref, pair.query)
    rms = matched_rms(soft, ref_free, pairs)
    extra = ""
    if pair.id == "hexane_cyclohexane":
        extra = f"; hexagon_rel_dev={hexagon_rel_dev(soft):.3f}"
    print(f"{pair.id}: mapped {len(pairs)}; soft RMS={rms:.3f}{extra}")
    show_row(
        f"{pair.id}: {pair.notes}",
        [
            ("ref free", paint_svg(ref, ref_free, title="ref")),
            ("query free", paint_svg(query, query_free, title="query")),
            ("query constrained", paint_svg(query, soft, title="constrained")),
        ],
    )
